# DeepDetect Experiments V2: vit_b_32
V2 pipeline: v5 skin-tone patch + phase FFT + original cleaner + dual transforms.
Spatial branch: full degradation augmentations.
Frequency branch: spatial aug only — patch selected from clean image.

Best hyperparameters from Optuna tuning (trial 13):
  backbone_lr=4.85e-05, lr=2.79e-04, freq_aux_weight=0.500,
  diversity_weight=0.058, gate_init_bias=0.215


In [1]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))
import torch
print(f"Device: {'cuda' if torch.cuda.is_available() else 'cpu'}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")


Device: cuda
GPU: NVIDIA RTX PRO 4000 Blackwell
VRAM: 25.2 GB


## Best Hyperparameters

In [2]:
BEST_BACKBONE_LR       = 4.85e-05
BEST_LR                = 2.79e-04
BEST_FREQ_AUX_WEIGHT   = 0.500
BEST_DIVERSITY_WEIGHT  = 0.058
BEST_GATE_INIT_BIAS    = 0.215


## Shared Config

In [3]:
from config import Config
from data.deepdetect_dual import get_deepdetect_dual_loaders
from experiments.train import train_v2
from experiments.evaluate import full_evaluation_v2
from experiments.baseline_spatial_only import run_spatial_only_baseline

BACKBONE = "vit_b_32"

def make_cfg(fusion_mode, frozen=False):
    cfg = Config()
    cfg.data.deepdetect_root      = "../data/raw/deep_detect/data"
    cfg.data.dataset              = "deepdetect"
    cfg.data.image_size           = 224
    cfg.data.batch_size           = 88
    cfg.data.num_workers          = 4
    cfg.backbone.name             = BACKBONE
    cfg.backbone.pretrained       = True
    cfg.backbone.img_size         = 224
    cfg.backbone.frozen           = frozen
    cfg.fusion.mode               = fusion_mode
    cfg.frequency.use_v2_pipeline = True
    cfg.frequency.cleaner_filters = 3
    cfg.frequency.patch_size      = 56
    cfg.train.early_stopping_patience = 30  
    cfg.train.epochs              = 35
    cfg.train.backbone_lr         = BEST_BACKBONE_LR
    cfg.train.lr                  = BEST_LR
    cfg.loss.freq_aux_weight      = BEST_FREQ_AUX_WEIGHT
    cfg.fusion.diversity_weight   = BEST_DIVERSITY_WEIGHT
    cfg.fusion.gate_init_bias     = BEST_GATE_INIT_BIAS
    # Spatial branch — full augmentations
    cfg.data.jpeg_aug             = True
    cfg.data.blur_aug             = True
    cfg.data.noise_aug            = True
    cfg.data.recompression_aug    = True
    cfg.data.mixed_aug            = True
    cfg.data.mixed_aug_prob       = 0.3
    cfg.experiment_name = f"dd_v2_{BACKBONE}_{fusion_mode}{'_frozen' if frozen else ''}"
    cfg.notes           = f"DeepDetect V2 · {BACKBONE} · {fusion_mode}{'_frozen' if frozen else ''}"
    return cfg

cfg_data = make_cfg("joint_only")
train_loader, val_loader, test_loader = get_deepdetect_dual_loaders(cfg_data)


Train: 76,848  Val: 13,561  Test: 21,776


## Experiment 0: Spatial-Only Baseline

In [4]:
cfg0 = make_cfg("joint_only")
cfg0.experiment_name = f"dd_v2_{BACKBONE}_spatial_only"
cfg0.notes           = f"DeepDetect V2 spatial-only baseline"
cfg0.train.epochs    = 20
spatial_acc = run_spatial_only_baseline(cfg0, train_loader, val_loader, test_loader)
print(f"\nSpatial-only floor: {spatial_acc:.1%}")


Device: cuda


Training spatial-only baseline (vit_b_32) for 20 epochs...
Train: 76,848  Val: 13,561


Epoch   1/20 | train_loss=0.2574 | val_acc=95.2%


Epoch   2/20 | train_loss=0.1187 | val_acc=95.2%


Epoch   3/20 | train_loss=0.0853 | val_acc=97.5%


...

Epoch  16/20 | train_loss=0.0024 | val_acc=99.0%


Epoch  17/20 | train_loss=0.0020 | val_acc=99.1%


Epoch  18/20 | train_loss=0.0017 | val_acc=99.1%


Epoch  19/20 | train_loss=0.0009 | val_acc=99.2%


Epoch  20/20 | train_loss=0.0010 | val_acc=99.2%

Spatial-only results (vit_b_32):
  Accuracy: 96.1%
  AUC-ROC:  0.983
  F1:       0.961
Results saved → ./results/results.csv  (dd_v2_vit_b_32_spatial_only, acc=0.9615)

Spatial-only floor: 96.1%


## Experiment 1: Joint-Only

In [5]:
cfg1 = make_cfg("joint_only")
train_v2(cfg1, train_loader, val_loader, test_loader)


Device: cuda
GPU: NVIDIA RTX PRO 4000 Blackwell
Using differential lr: backbone=4.85e-05, others=2.79e-04

Experiment: dd_v2_vit_b_32_joint_only [V2 pipeline]
Backbone: vit_b_32 | Fusion: joint_only | Frozen: False
Epochs: 35 | LR: 0.000279 | Batch: 88
Spatial branch: full aug | Frequency branch: spatial aug only



Epoch   1/35 | train_loss=0.5927 | val_acc=91.1% | val_auc=0.997 | val_f1=0.893 | best=0.0% | patience=0/30
  -> Saved checkpoint (epoch 1, val_acc=91.1%)


Epoch   2/35 | train_loss=0.4410 | val_acc=97.6% | val_auc=0.998 | val_f1=0.974 | best=91.1% | patience=0/30
  -> Saved checkpoint (epoch 2, val_acc=97.6%)


Epoch   3/35 | train_loss=0.3962 | val_acc=93.9% | val_auc=0.998 | val_f1=0.929 | best=97.6% | patience=0/30
  -> Saved checkpoint (epoch 3, val_acc=93.9%)


Epoch   4/35 | train_loss=0.3734 | val_acc=97.3% | val_auc=0.998 | val_f1=0.970 | best=97.6% | patience=0/30
  -> Saved checkpoint (epoch 4, val_acc=97.3%)


...

Epoch  32/35 | train_loss=0.2540 | val_acc=99.4% | val_auc=1.000 | val_f1=0.994 | best=99.4% | patience=0/30
  -> Saved checkpoint (epoch 32, val_acc=99.4%)


Epoch  33/35 | train_loss=0.2542 | val_acc=99.4% | val_auc=1.000 | val_f1=0.994 | best=99.4% | patience=0/30
  -> Saved checkpoint (epoch 33, val_acc=99.4%)


Epoch  34/35 | train_loss=0.2542 | val_acc=99.4% | val_auc=1.000 | val_f1=0.993 | best=99.4% | patience=0/30
  -> Saved checkpoint (epoch 34, val_acc=99.4%)


Epoch  35/35 | train_loss=0.2530 | val_acc=99.3% | val_auc=1.000 | val_f1=0.992 | best=99.4% | patience=0/30
  -> Saved checkpoint (epoch 35, val_acc=99.3%)

Training complete. Best val accuracy: 99.4%
Results will be logged to CSV after full_evaluation_v2().


0.9943219526583585

In [6]:
results1 = full_evaluation_v2(
    cfg1,
    checkpoint_path=f"./checkpoints/best_{cfg1.experiment_name}.pt",
    test_loader=test_loader,
    dataset_type="deepdetect",
)

Loaded checkpoint: ./checkpoints/best_dd_v2_vit_b_32_joint_only.pt

EVALUATION — dd_v2_vit_b_32_joint_only
Backbone: vit_b_32 | Fusion: joint_only | Frozen: False
  Joint accuracy:     96.6%
  AUC-ROC:            0.988
  F1:                 0.965
  Spatial-only:       96.5%
  Freq-only:          73.3%
  Delta (Δ):          +0.1%  (freq branch contribution)

  No warning signs triggered.
Results saved → ./results/results.csv  (dd_v2_vit_b_32_joint_only, acc=0.9656)



## Experiment 2: Scalar Fusion

In [7]:
cfg2 = make_cfg("scalar")
train_v2(cfg2, train_loader, val_loader, test_loader)

Device: cuda
GPU: NVIDIA RTX PRO 4000 Blackwell
Using differential lr: backbone=4.85e-05, others=2.79e-04

Experiment: dd_v2_vit_b_32_scalar [V2 pipeline]
Backbone: vit_b_32 | Fusion: scalar | Frozen: False
Epochs: 35 | LR: 0.000279 | Batch: 88
Spatial branch: full aug | Frequency branch: spatial aug only



Epoch   1/35 | train_loss=0.5930 | val_acc=92.1% | val_auc=0.996 | val_f1=0.906 | best=0.0% | patience=0/30
  -> Saved checkpoint (epoch 1, val_acc=92.1%)


Epoch   2/35 | train_loss=0.4354 | val_acc=96.9% | val_auc=0.998 | val_f1=0.966 | best=92.1% | patience=0/30
  -> Saved checkpoint (epoch 2, val_acc=96.9%)


Epoch   3/35 | train_loss=0.3923 | val_acc=97.8% | val_auc=0.999 | val_f1=0.975 | best=96.9% | patience=0/30
  -> Saved checkpoint (epoch 3, val_acc=97.8%)


Epoch   4/35 | train_loss=0.3727 | val_acc=97.7% | val_auc=0.999 | val_f1=0.975 | best=97.8% | patience=0/30
  -> Saved checkpoint (epoch 4, val_acc=97.7%)


...

Epoch  32/35 | train_loss=0.2553 | val_acc=99.3% | val_auc=1.000 | val_f1=0.993 | best=99.3% | patience=0/30
  -> Saved checkpoint (epoch 32, val_acc=99.3%)


Epoch  33/35 | train_loss=0.2557 | val_acc=99.3% | val_auc=1.000 | val_f1=0.992 | best=99.3% | patience=0/30
  -> Saved checkpoint (epoch 33, val_acc=99.3%)


Epoch  34/35 | train_loss=0.2559 | val_acc=99.3% | val_auc=1.000 | val_f1=0.992 | best=99.3% | patience=0/30
  -> Saved checkpoint (epoch 34, val_acc=99.3%)


Epoch  35/35 | train_loss=0.2547 | val_acc=99.3% | val_auc=1.000 | val_f1=0.992 | best=99.3% | patience=0/30
  -> Saved checkpoint (epoch 35, val_acc=99.3%)

Training complete. Best val accuracy: 99.3%
Results will be logged to CSV after full_evaluation_v2().


0.9934370621635572

In [8]:
results2 = full_evaluation_v2(
    cfg2,
    checkpoint_path=f"./checkpoints/best_{cfg2.experiment_name}.pt",
    test_loader=test_loader,
    dataset_type="deepdetect",
)

Loaded checkpoint: ./checkpoints/best_dd_v2_vit_b_32_scalar.pt

EVALUATION — dd_v2_vit_b_32_scalar
Backbone: vit_b_32 | Fusion: scalar | Frozen: False
  Joint accuracy:     96.2%
  AUC-ROC:            0.988
  F1:                 0.961
  Spatial-only:       96.2%
  Freq-only:          71.9%
  Delta (Δ):          -0.0%  (freq branch contribution)
Results saved → ./results/results.csv  (dd_v2_vit_b_32_scalar, acc=0.9618)



## Experiment 3: Gating Fusion

In [9]:
cfg3 = make_cfg("gating")
train_v2(cfg3, train_loader, val_loader, test_loader)


Device: cuda
GPU: NVIDIA RTX PRO 4000 Blackwell
Using differential lr: backbone=4.85e-05, others=2.79e-04

Experiment: dd_v2_vit_b_32_gating [V2 pipeline]
Backbone: vit_b_32 | Fusion: gating | Frozen: False
Epochs: 35 | LR: 0.000279 | Batch: 88
Spatial branch: full aug | Frequency branch: spatial aug only



Epoch   1/35 | train_loss=0.6429 | val_acc=97.2% | val_auc=0.997 | val_f1=0.969 | best=0.0% | patience=0/30
  -> Saved checkpoint (epoch 1, val_acc=97.2%)


Epoch   2/35 | train_loss=0.4770 | val_acc=96.9% | val_auc=0.997 | val_f1=0.966 | best=97.2% | patience=0/30
  -> Saved checkpoint (epoch 2, val_acc=96.9%)


Epoch   3/35 | train_loss=0.4258 | val_acc=97.9% | val_auc=0.999 | val_f1=0.977 | best=97.2% | patience=0/30
  -> Saved checkpoint (epoch 3, val_acc=97.9%)


Epoch   4/35 | train_loss=0.3970 | val_acc=98.2% | val_auc=0.998 | val_f1=0.980 | best=97.9% | patience=0/30
  -> Saved checkpoint (epoch 4, val_acc=98.2%)


...

Epoch  32/35 | train_loss=0.3046 | val_acc=99.3% | val_auc=1.000 | val_f1=0.992 | best=99.3% | patience=0/30
  -> Saved checkpoint (epoch 32, val_acc=99.3%)


Epoch  33/35 | train_loss=0.3080 | val_acc=99.2% | val_auc=1.000 | val_f1=0.991 | best=99.3% | patience=0/30
  -> Saved checkpoint (epoch 33, val_acc=99.2%)


Epoch  34/35 | train_loss=0.3068 | val_acc=99.3% | val_auc=1.000 | val_f1=0.992 | best=99.3% | patience=0/30
  -> Saved checkpoint (epoch 34, val_acc=99.3%)


Epoch  35/35 | train_loss=0.3062 | val_acc=99.3% | val_auc=1.000 | val_f1=0.992 | best=99.3% | patience=0/30
  -> Saved checkpoint (epoch 35, val_acc=99.3%)

Training complete. Best val accuracy: 99.3%
Results will be logged to CSV after full_evaluation_v2().


0.9933633212889905

In [10]:
results3 = full_evaluation_v2(
    cfg3,
    checkpoint_path=f"./checkpoints/best_{cfg3.experiment_name}.pt",
    test_loader=test_loader,
    dataset_type="deepdetect",
)


Loaded checkpoint: ./checkpoints/best_dd_v2_vit_b_32_gating.pt

EVALUATION — dd_v2_vit_b_32_gating
Backbone: vit_b_32 | Fusion: gating | Frozen: False
  Joint accuracy:     96.0%
  AUC-ROC:            0.982
  F1:                 0.959
  Spatial-only:       95.9%
  Freq-only:          72.1%
  Delta (Δ):          +0.1%  (freq branch contribution)

  Gate distribution:
    entropy:  1.263 nats  (OK)
    mean:     0.676
    variance: 0.0021

  No warning signs triggered.
Results saved → ./results/results.csv  (dd_v2_vit_b_32_gating, acc=0.9599)



## Experiment 4: Gating Frozen

In [11]:
cfg4 = make_cfg("gating", frozen=True)
train_v2(cfg4, train_loader, val_loader, test_loader)


Device: cuda
GPU: NVIDIA RTX PRO 4000 Blackwell
Using differential lr: backbone=4.85e-05, others=2.79e-04

Experiment: dd_v2_vit_b_32_gating_frozen [V2 pipeline]
Backbone: vit_b_32 | Fusion: gating | Frozen: True
Epochs: 35 | LR: 0.000279 | Batch: 88
Spatial branch: full aug | Frequency branch: spatial aug only



Epoch   1/35 | train_loss=0.8907 | val_acc=81.5% | val_auc=0.924 | val_f1=0.766 | best=0.0% | patience=0/30
  -> Saved checkpoint (epoch 1, val_acc=81.5%)


Epoch   2/35 | train_loss=0.7686 | val_acc=76.2% | val_auc=0.932 | val_f1=0.660 | best=81.5% | patience=0/30
  -> Saved checkpoint (epoch 2, val_acc=76.2%)


Epoch   3/35 | train_loss=0.7188 | val_acc=81.3% | val_auc=0.953 | val_f1=0.751 | best=81.5% | patience=0/30
  -> Saved checkpoint (epoch 3, val_acc=81.3%)


Epoch   4/35 | train_loss=0.6885 | val_acc=83.0% | val_auc=0.955 | val_f1=0.781 | best=81.5% | patience=0/30
  -> Saved checkpoint (epoch 4, val_acc=83.0%)


...

Epoch  32/35 | train_loss=0.4542 | val_acc=87.3% | val_auc=0.972 | val_f1=0.844 | best=89.6% | patience=0/30
  -> Saved checkpoint (epoch 32, val_acc=87.3%)


Epoch  33/35 | train_loss=0.4497 | val_acc=87.7% | val_auc=0.972 | val_f1=0.850 | best=89.6% | patience=0/30
  -> Saved checkpoint (epoch 33, val_acc=87.7%)


Epoch  34/35 | train_loss=0.4500 | val_acc=87.5% | val_auc=0.972 | val_f1=0.848 | best=89.6% | patience=0/30
  -> Saved checkpoint (epoch 34, val_acc=87.5%)


Epoch  35/35 | train_loss=0.4499 | val_acc=87.6% | val_auc=0.973 | val_f1=0.849 | best=89.6% | patience=0/30
  -> Saved checkpoint (epoch 35, val_acc=87.6%)

Training complete. Best val accuracy: 89.6%
Results will be logged to CSV after full_evaluation_v2().


0.8958778851117174

In [12]:
results4 = full_evaluation_v2(
    cfg4,
    checkpoint_path=f"./checkpoints/best_{cfg4.experiment_name}.pt",
    test_loader=test_loader,
    dataset_type="deepdetect",
)


Loaded checkpoint: ./checkpoints/best_dd_v2_vit_b_32_gating_frozen.pt

EVALUATION — dd_v2_vit_b_32_gating_frozen
Backbone: vit_b_32 | Fusion: gating | Frozen: True
  Joint accuracy:     85.0%
  AUC-ROC:            0.948
  F1:                 0.828
  Spatial-only:       75.1%
  Freq-only:          71.3%
  Delta (Δ):          +10.0%  (freq branch contribution)

  Gate distribution:
    entropy:  2.891 nats  (OK)
    mean:     0.594
    variance: 0.0745

  No warning signs triggered.
Results saved → ./results/results.csv  (dd_v2_vit_b_32_gating_frozen, acc=0.8502)



## Summary

In [15]:
import pandas as pd
df = pd.read_csv("./results/results.csv",  on_bad_lines='skip')
mask = df["experiment_name"].str.startswith(f"dd_v2_{BACKBONE}")
cols = ["experiment_name", "fusion", "frozen", "accuracy", "auc_roc", "f1", "gate_entropy"]
print(df[mask][cols].to_string(index=False))


             experiment_name     fusion  frozen  accuracy  auc_roc     f1  gate_entropy
 dd_v2_vit_b_32_spatial_only joint_only   False    0.9615   0.9834 0.9608        0.0000
   dd_v2_vit_b_32_joint_only joint_only   False    0.9656   0.9882 0.9648        0.0000
       dd_v2_vit_b_32_scalar     scalar   False    0.9618   0.9882 0.9611        0.0000
       dd_v2_vit_b_32_gating     gating   False    0.9599   0.9819 0.9593        1.2635
dd_v2_vit_b_32_gating_frozen     gating    True    0.8502   0.9481 0.8279        2.8911
